In [3]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

DATA_DIR = '/content/drive/MyDrive/머신러닝/data'

# ============================================
# 1. 저장된 파일 로드
# ============================================
tab_oof = np.load(f'{DATA_DIR}/oof_tabnet.npy')
relu_oof = np.load(f'{DATA_DIR}/oof_relu.npy')
gelu_oof = np.load(f'{DATA_DIR}/oof_gelu.npy')

tab_test = np.load(f'{DATA_DIR}/test_tabnet.npy')
relu_test = np.load(f'{DATA_DIR}/test_relu.npy')
gelu_test = np.load(f'{DATA_DIR}/test_gelu.npy')
ag_test = np.load(f'{DATA_DIR}/test_autogluon.npy')

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
y = train['target'].values.astype(int)

print("=" * 60)
print("🔍 진단 1: shape & 데이터 정합성")
print("=" * 60)
print(f"OOF shapes:")
print(f"  TabNet:   {tab_oof.shape}")
print(f"  MLP-ReLU: {relu_oof.shape}")
print(f"  MLP-GELU: {gelu_oof.shape}")
print(f"\nTest shapes:")
print(f"  TabNet:   {tab_test.shape}")
print(f"  MLP-ReLU: {relu_test.shape}")
print(f"  MLP-GELU: {gelu_test.shape}")
print(f"  AutoGluon: {ag_test.shape}")
print(f"\ny shape: {y.shape}")
print(f"test shape: {test.shape}")

# ============================================
# 2. OOF 정합성 (각 행 합이 1인가?)
# ============================================
print("\n" + "=" * 60)
print("🔍 진단 2: OOF 확률 합 = 1?")
print("=" * 60)
print(f"TabNet OOF 행 합 평균: {tab_oof.sum(axis=1).mean():.4f}")
print(f"ReLU OOF 행 합 평균:   {relu_oof.sum(axis=1).mean():.4f}")
print(f"GELU OOF 행 합 평균:   {gelu_oof.sum(axis=1).mean():.4f}")
print(f"TabNet Test 행 합 평균:{tab_test.sum(axis=1).mean():.4f}")
print(f"AG Test 행 합 평균:    {ag_test.sum(axis=1).mean():.4f}")

# ============================================
# 3. OOF 점수 재확인 (각 모델)
# ============================================
print("\n" + "=" * 60)
print("🔍 진단 3: OOF 점수 재확인")
print("=" * 60)
print(f"TabNet OOF F1:   {f1_score(y, tab_oof.argmax(1), average='macro'):.4f}")
print(f"MLP-ReLU OOF F1: {f1_score(y, relu_oof.argmax(1), average='macro'):.4f}")
print(f"MLP-GELU OOF F1: {f1_score(y, gelu_oof.argmax(1), average='macro'):.4f}")
print(f"3명 균등 OOF F1: {f1_score(y, ((tab_oof+relu_oof+gelu_oof)/3).argmax(1), average='macro'):.4f}")

# ============================================
# 4. Test 예측의 클래스 분포 (Train과 비슷한가?)
# ============================================
print("\n" + "=" * 60)
print("🔍 진단 4: Test 예측 분포 (각 모델 단독)")
print("=" * 60)
print(f"{'클래스':>6} {'TabNet':>8} {'ReLU':>8} {'GELU':>8} {'AG':>8}")
for c in range(21):
    tab_c = (tab_test.argmax(1) == c).sum()
    relu_c = (relu_test.argmax(1) == c).sum()
    gelu_c = (gelu_test.argmax(1) == c).sum()
    ag_c = (ag_test.argmax(1) == c).sum()
    print(f"{c:>6} {tab_c:>8} {relu_c:>8} {gelu_c:>8} {ag_c:>8}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 진단 1: shape & 데이터 정합성
OOF shapes:
  TabNet:   (21693, 21)
  MLP-ReLU: (21693, 21)
  MLP-GELU: (21693, 21)

Test shapes:
  TabNet:   (15004, 21)
  MLP-ReLU: (15004, 21)
  MLP-GELU: (15004, 21)
  AutoGluon: (15004, 21)

y shape: (21693,)
test shape: (15004, 53)

🔍 진단 2: OOF 확률 합 = 1?
TabNet OOF 행 합 평균: 1.0000
ReLU OOF 행 합 평균:   1.0000
GELU OOF 행 합 평균:   1.0000
TabNet Test 행 합 평균:1.0000
AG Test 행 합 평균:    1.0000

🔍 진단 3: OOF 점수 재확인
TabNet OOF F1:   0.8541
MLP-ReLU OOF F1: 0.8593
MLP-GELU OOF F1: 0.8637
3명 균등 OOF F1: 0.8709

🔍 진단 4: Test 예측 분포 (각 모델 단독)
   클래스   TabNet     ReLU     GELU       AG
     0      739     1028      991      840
     1      509      711      690      468
     2      417      416      491      416
     3      963     1078     1058      979
     4      739      735      736      729
     5      443      451      453      440
     6      

In [4]:
# AutoGluon이 다른 모델 의견 덮어버린다고 의심 → 가중치 낮춤
ensemble = 0.30*tab_test + 0.30*relu_test + 0.30*gelu_test + 0.10*ag_test
preds = ensemble.argmax(axis=1)

submission = pd.DataFrame({"ID": test["ID"], "target": preds})
submission.to_csv(f'{DATA_DIR}/submission_ag_low.csv', index=False)
print("✅ Saved with AG weight 0.10")

✅ Saved with AG weight 0.10


In [5]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '/content/drive/MyDrive/머신러닝/data'

# 저장된 확률 로드
tab_test = np.load(f'{DATA_DIR}/test_tabnet.npy')
relu_test = np.load(f'{DATA_DIR}/test_relu.npy')
gelu_test = np.load(f'{DATA_DIR}/test_gelu.npy')
ag_test = np.load(f'{DATA_DIR}/test_autogluon.npy')

test = pd.read_csv(f'{DATA_DIR}/test.csv')

print(f"✅ Loaded")
print(f"  Test shapes: {tab_test.shape}")
print(f"  test.csv: {test.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Loaded
  Test shapes: (15004, 21)
  test.csv: (15004, 53)


In [6]:
def compute_quota(probs):
    N, C = probs.shape
    base = N // C
    rem = N % C
    quota = np.full(C, base, int)
    order = np.argsort(-probs.mean(0))
    quota[order[:rem]] += 1
    return quota

def balanced_assign_hungarian(probs, quota):
    eps = 1e-15
    N, C = probs.shape
    cols, col_to_class = [], []
    for c in range(C):
        q = int(quota[c])
        cost_c = -np.log(probs[:,c] + eps)
        cols.append(np.repeat(cost_c[:,None], q, axis=1))
        col_to_class.extend([c]*q)
    cost = np.concatenate(cols, axis=1)
    row_ind, col_ind = linear_sum_assignment(cost)
    pred = np.zeros(N, int)
    pred[row_ind] = np.array(col_to_class)[col_ind]
    return pred

print("✅ 후처리 함수 정의 완료")

✅ 후처리 함수 정의 완료


In [7]:
# 시도 1: 4명 균등 + Hungarian
ensemble = 0.25*tab_test + 0.25*relu_test + 0.25*gelu_test + 0.25*ag_test
quota = compute_quota(ensemble)
preds = balanced_assign_hungarian(ensemble, quota)

submission = pd.DataFrame({"ID": test["ID"], "target": preds})
output_path = f'{DATA_DIR}/submission_try1_4models_hung.csv'
submission.to_csv(output_path, index=False)

print(f"✅ 시도 1 저장: {output_path}")
print(f"클래스 분포:")
print(pd.Series(preds).value_counts().sort_index().head(10))

✅ 시도 1 저장: /content/drive/MyDrive/머신러닝/data/submission_try1_4models_hung.csv
클래스 분포:
0    715
1    714
2    714
3    715
4    715
5    714
6    714
7    714
8    715
9    715
Name: count, dtype: int64


In [9]:
# 시도 2: AG 가중치 0.10, 나머지 0.30씩 + Argmax
ensemble = 0.30*tab_test + 0.30*relu_test + 0.30*gelu_test + 0.10*ag_test
preds = ensemble.argmax(axis=1)

submission = pd.DataFrame({"ID": test["ID"], "target": preds})
output_path = f'{DATA_DIR}/submission_try2_ag_low.csv'
submission.to_csv(output_path, index=False)

print(f"✅ 시도 2 저장: {output_path}")
print(f"클래스 분포:")
print(pd.Series(preds).value_counts().sort_index().head(10))

✅ 시도 2 저장: /content/drive/MyDrive/머신러닝/data/submission_try2_ag_low.csv
클래스 분포:
0     924
1     679
2     421
3    1033
4     736
5     450
6     714
7     464
8    1051
9     330
Name: count, dtype: int64


In [10]:
# 시도 3: 3명 균등 (AG 제외) + Hungarian
ensemble = (tab_test + relu_test + gelu_test) / 3
quota = compute_quota(ensemble)
preds = balanced_assign_hungarian(ensemble, quota)

submission = pd.DataFrame({"ID": test["ID"], "target": preds})
output_path = f'{DATA_DIR}/submission_try3_3models_hung.csv'
submission.to_csv(output_path, index=False)

print(f"✅ 시도 3 저장: {output_path}")
print(f"클래스 분포:")
print(pd.Series(preds).value_counts().sort_index().head(10))

✅ 시도 3 저장: /content/drive/MyDrive/머신러닝/data/submission_try3_3models_hung.csv
클래스 분포:
0    715
1    714
2    714
3    715
4    715
5    714
6    714
7    714
8    715
9    715
Name: count, dtype: int64
